# Scénario 11d : les entrées en retard, trois spécialistes

Journal : [`tableau_resultats.md`](tableau_resultats.md). Essai **bonus et indépendant** des autres branches de machine learning du projet.

**Question** : parmi les clients **sains en août** (-2, -1 ou 0), lesquels **entrent en retard** en septembre (2 et plus) ? C'est le pendant du `11c` (sorties du retard). Détecter un retard compte : il annonce les clients qui risquent de faire défaut.

**Données** : `data/creditcard_pret_ingestion.csv` (niveau 0), sans `dpnm`, sans `ID`, sans données personnelles, **sans correctifs**. Même découpage de septembre que les notebooks 01 à 11c (clients dont `PAY_1` n'est pas un 1, 80/20, `stratify` sur `PAY_1`, `random_state=42`) ; le test n'est pas lu.

**Exemples** : **septembre seul, sans empilement**, avec **tout l'historique** d'avril à septembre (décision de l'utilisateur) : les variables du `11b` (codifications d'avril à août regroupées en -2, -1, 0, 2 et plus ; soldes et paiements d'avril à septembre ; plafond) ; **aucune colonne ajoutée**. Population : tous les clients du train sains en août, sans autre restriction.

**Cible** : 1 = **entrée en retard** (`PAY_1` >= 2) ; 0 = reste sain.

**Trois spécialistes** : **MLP** (moyenne de 5 graines), **CatBoost** et **RandomForest** (meilleure précision sur les nouveaux retards au passage 13). Boucle d'optimisation de l'essai (fenêtre de trois valeurs, 10 tours au plus, surapprentissage limité à 7 % relatif, alerte si l'écart-type des plis dépasse 2 points), départ aux meilleurs réglages du `11b` ; critère : **précision moyenne sur les entrées en retard**.

**Seuil de décision : précision marginale de 50 %**, fixée à l'avance par l'utilisateur : chaque modèle annonce des entrées en retard tant que, parmi les 50 derniers annoncés, au moins 1 sur 2 entre réellement en retard. Le seuil est calculé dans des plis internes, puis jugé sur le pli de validation. **Décision commune : 2 voix sur 3** ; l'unanimité et « au moins un » sont affichés pour comparaison.

**Attendu** : au passage 13, les modèles généralistes ne trouvaient qu'une petite part des nouveaux retards, avec beaucoup de fausses alertes ; le paiement d'octobre, qui décide de la codification de septembre, manque. Peu d'entrées annoncées, voire aucune, serait un résultat en soi.

**Limites** : comme au `11c`, la précision mesurée sur les clients tranchés n'est pas garantie pour les `PAY_1` = 1, et le modèle apprend la codification habituelle de la banque (fiabilité de `PAY_1` non vérifiée).

## 1. Données, découpage de septembre et population

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)

mask_1 = df["PAY_1"] == 1
connus, a_trancher = df[~mask_1], df[mask_1]
X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)

def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 (et 1) inchangés
    return codifications.where(codifications < 2, 2)

# Population : clients du train sains en août ; cible : entrée en retard en septembre
sains = X_train["PAY_2"] <= 0
X = X_train[sains].copy()
for col in CATEGORIELLES:
    X[col] = regrouper(X[col])
y = (y_train[sains] >= 2).astype(int)
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}, non lu)")
print(f"Clients du train sains en août : {len(X)} ; entrées en retard en septembre : {int(y.sum())} ({y.mean() * 100:.1f} %)")
display(pd.crosstab(X_train.loc[sains, "PAY_2"].rename("Août (PAY_2)"), y.map({0: "reste sain", 1: "entre en retard"}).rename("Septembre"), margins=True, margins_name="Total"))
print(f"PAY_1 = 1 sains en août : {int((a_trancher['PAY_2'] <= 0).sum())} (dont -2 : {int((a_trancher['PAY_2'] == -2).sum())}, "
      f"-1 : {int((a_trancher['PAY_2'] == -1).sum())}, 0 : {int((a_trancher['PAY_2'] == 0).sum())})")

PAY_1 connus : 26312 (train 21049, test 5263, non lu)
Clients du train sains en août : 18978 ; entrées en retard en septembre : 799 (4.2 %)


Septembre,entre en retard,reste sain,Total
Août (PAY_2),,,
-2,0,2051,2051
-1,95,4234,4329
0,704,11894,12598
Total,799,18179,18978


PAY_1 = 1 sains en août : 1836 (dont -2 : 1221, -1 : 612, 0 : 3)


**Comment lire** : le tableau croise la codification d'août des clients sains (en ligne) et leur situation en septembre. La dernière ligne donne la population des `PAY_1` = 1 à laquelle les modèles seront appliqués (section 7).

## 2. Plis, critère et modèles

**Plis** : 5 plis stratifiés sur les entrées en retard (`random_state=42`). **Critère** : la précision moyenne sur les entrées en retard (vaut leur part pour un tri au hasard).

**Modèles** :
- **MLP** : moyenne des probabilités de 5 MLP de graines différentes ; couches cachées et `alpha` explorés ; arrêt anticipé ;
- **CatBoost** : codifications en une colonne par valeur, tous les cœurs, 1 000 itérations, perte `Logloss` ; profondeur, `learning_rate`, `l2_leaf_reg` explorés ; poids des classes `Balanced` ou `SqrtBalanced` ;
- **RandomForest** : nombre d'arbres, profondeur (plafonnée à 30), taille minimale des feuilles explorés ; `max_features` et poids des classes en choix.

In [3]:
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Logarithme signé des soldes, des paiements et du plafond
def log_montants(X):
    X = X.copy()
    X[NUMERIQUES] = np.sign(X[NUMERIQUES]) * np.log1p(np.abs(X[NUMERIQUES]))
    return X

def preprocesseur_lineaire():
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

class MoyenneGraines(ClassifierMixin, BaseEstimator):
    # Moyenne des probabilités de plusieurs copies du même modèle, chacune avec sa graine
    def __init__(self, modele=None, graines=5, parametre_graine="mlpclassifier__random_state"):
        self.modele, self.graines, self.parametre_graine = modele, graines, parametre_graine

    def fit(self, X, y):
        self.modeles_ = [clone(self.modele).set_params(**{self.parametre_graine: g}).fit(X, y) for g in range(self.graines)]
        self.classes_ = self.modeles_[0].classes_
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.modeles_], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))

ESPACES = {
    "MLP": {
        "modele": MoyenneGraines(make_pipeline(FunctionTransformer(log_montants), preprocesseur_lineaire(),
                                               MLPClassifier(max_iter=500, early_stopping=True, random_state=0))),
        "echelles": {"modele__mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "modele__mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"modele__mlpclassifier__hidden_layer_sizes": (256, 128), "modele__mlpclassifier__alpha": 1e-3},   # meilleur réglage du 11b
        "choix": {},
    },
    "CatBoost": {
        "modele": make_pipeline(FunctionTransformer(log_montants), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="Logloss", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 4, "catboostclassifier__learning_rate": 0.03, "catboostclassifier__l2_leaf_reg": 10},   # meilleur réglage du 11b
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
    "RandomForest": {
        "modele": make_pipeline(FunctionTransformer(log_montants), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, 30],   # plafond à 30
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 500, "randomforestclassifier__max_depth": 20, "randomforestclassifier__min_samples_leaf": 50},   # meilleur réglage du 11b
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES), f"; précision moyenne d'un tri au hasard (part des entrées en retard) : {y.mean() * 100:.1f} %")

Modèles : ['MLP', 'CatBoost', 'RandomForest'] ; précision moyenne d'un tri au hasard (part des entrées en retard) : 4.2 %


## 3. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch sur les 5 plis, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Une ligne par tour, puis une ligne par modèle.

In [4]:
def fenetre(echelle, position):
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[-1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        debut_tour = time.perf_counter()
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring="average_precision", cv=cv, n_jobs=-1, return_train_score=True, refit=False)
        recherche.fit(X, y)
        r = recherche.cv_results_
        res = pd.DataFrame({"tour": tour, "params": r["params"], "validation": r["mean_test_score"] * 100,
                            "écart-type des plis": r["std_test_score"] * 100, "train": r["mean_train_score"] * 100})
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            print(f"{nom}, tour {tour} : {len(res)} réglages, aucun admis ({round(time.perf_counter() - debut_tour)} s)", flush=True)
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        print(f"{nom}, tour {tour} : {len(res)} réglages, {len(admis)} admis ; meilleur admis {admis['validation'].max():.2f} % "
              f"({libelle(meilleur)}) ({round(time.perf_counter() - debut_tour)} s)", flush=True)
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    return pd.concat(lignes, ignore_index=True), raison, round(time.perf_counter() - debut)

resultats, lignes, meilleurs = {}, [], {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        ligne, alerte = historique.loc[historique["surapprentissage (% relatif)"].idxmin()], "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    print(f"{nom} : {ligne['validation']:.2f} % ± {ligne['écart-type des plis']:.2f} ({libelle(ligne['params'])}) ; arrêt : {raison} ({duree} s)")
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   "Précision moyenne sur les entrées, validation (%)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2), "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(historique["tour"].max()), "Arrêt": raison, "Durée (s)": duree})
display(pd.DataFrame(lignes).set_index("Modèle"))

MLP, tour 1 : 9 réglages, 9 admis ; meilleur admis 12.66 % (alpha=0.001, hidden_layer_sizes=(256, 128, 64)) (29 s)
MLP, tour 2 : 6 réglages, 6 admis ; meilleur admis 12.66 % (alpha=0.001, hidden_layer_sizes=(256, 128, 64)) (24 s)
MLP : 12.66 % ± 1.15 (alpha=0.001, hidden_layer_sizes=(256, 128, 64)) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (53 s)
CatBoost, tour 1 : 54 réglages, aucun admis (236 s)
CatBoost : 13.72 % ± 0.71 (auto_class_weights=Balanced, depth=3, l2_leaf_reg=15, learning_rate=0.02) ; arrêt : tour 1 : aucun réglage sous 7.0 % de surapprentissage (236 s)
RandomForest, tour 1 : 72 réglages, aucun admis (247 s)
RandomForest : 14.00 % ± 1.53 (class_weight=None, max_depth=15, max_features=sqrt, min_samples_leaf=50, n_estimators=300) ; arrêt : tour 1 : aucun réglage sous 7.0 % de surapprentissage (247 s)


,Réglage,"Précision moyenne sur les entrées, validation (%)",Train (%),Surapprentissage (% relatif),Alerte,Tours,Arrêt,Durée (s)
Modèle,,,,,,,,
MLP,"alpha=0.001, hidden_layer_sizes=(256, 128, 64)",12.66 ± 1.15,12.42,-1.92,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,53
CatBoost,"auto_class_weights=Balanced, depth=3, l2_leaf_...",13.72 ± 0.71,21.64,36.60,⚠️ aucun réglage admis (le moins surappris est...,1,tour 1 : aucun réglage sous 7.0 % de surappren...,236
RandomForest,"class_weight=None, max_depth=15, max_features=...",14.00 ± 1.53,25.63,45.37,⚠️ aucun réglage admis (le moins surappris est...,1,tour 1 : aucun réglage sous 7.0 % de surappren...,247


**Comment lire** : la précision moyenne vaut la part des entrées en retard pour un tri au hasard, et 100 % pour un tri parfait. Elle ne dépend d'aucun seuil.

## 4. Seuil à 50 % de précision marginale et vote à 2 voix sur 3, pli par pli

Sur chaque pli : chaque modèle est réentraîné sur les 4 autres plis ; une **validation croisée interne** à ces 4 plis donne des probabilités hors pli, sur lesquelles son seuil est fixé (on descend dans la liste tant que, parmi les 50 derniers annoncés, au moins 1 sur 2 entre réellement en retard). Le pli de validation est jugé avec ces seuils. **Décision commune : une entrée en retard est annoncée si au moins 2 des 3 modèles l'annoncent.**

**Mesures** : entrées annoncées, **taux d'entrées retrouvées**, **précision des entrées annoncées**, et, tous plis réunis, la précision marginale en validation sur les 50 annoncés les plus proches du seuil (modèles seuls).

In [5]:
from sklearn.inspection import permutation_importance

PRECISION_MARGINALE_VISEE = 0.50      # fixée à l'avance par l'utilisateur
FENETRE_MARGE = 50                    # précision marginale mesurée sur les 50 derniers annoncés

def seuil_marginal(y_vrai, proba, visee=PRECISION_MARGINALE_VISEE, fenetre=FENETRE_MARGE):
    # Clients rangés du plus probablement entré en retard au moins probable ; on annonce tant que la part de vraies entrées
    # parmi les `fenetre` derniers annoncés reste >= visee. Renvoie la probabilité du dernier annoncé (inf : aucun)
    ordre = np.argsort(-proba, kind="mergesort")
    y_tri, p_tri = np.asarray(y_vrai)[ordre], proba[ordre]
    if len(y_tri) < fenetre:
        return np.inf
    glissante = np.convolve(y_tri, np.ones(fenetre) / fenetre, mode="valid")
    sous = np.flatnonzero(glissante < visee - 1e-9)
    if len(sous) == 0:
        return p_tri[-1]
    if sous[0] == 0:
        return np.inf
    return p_tri[sous[0] + fenetre - 2]

def probas_internes(modele, apprentissage):
    # Validation croisée interne sur les clients d'apprentissage d'un pli : probabilités hors pli (sert à fixer le seuil)
    X_a, y_a = X.iloc[apprentissage], y.to_numpy()[apprentissage]
    p = np.full(len(y_a), np.nan)
    for app, val in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_a, y_a):
        p[val] = clone(modele).fit(X_a.iloc[app], y_a[app]).predict_proba(X_a.iloc[val])[:, 1]
    return y_a, p

y_np = y.to_numpy()
probas, seuils, annonces, mesures_auc, importances = {}, {}, {}, {}, {}
for nom, params in meilleurs.items():
    debut = time.perf_counter()
    modele = clone(ESPACES[nom]["modele"]).set_params(**params)
    probas[nom], annonces[nom] = np.zeros(len(y)), np.zeros(len(y), dtype=bool)
    seuils[nom], mesures_auc[nom], par_pli = [], {"Précision moyenne": [], "ROC AUC": []}, []
    for apprentissage, validation in cv:
        y_int, p_int = probas_internes(modele, apprentissage)
        s = seuil_marginal(y_int, p_int)
        m = clone(modele).fit(X.iloc[apprentissage], y_np[apprentissage])
        p = m.predict_proba(X.iloc[validation])[:, 1]
        probas[nom][validation], annonces[nom][validation] = p, p >= s
        seuils[nom].append(s)
        mesures_auc[nom]["Précision moyenne"].append(average_precision_score(y_np[validation], p))
        mesures_auc[nom]["ROC AUC"].append(roc_auc_score(y_np[validation], p))
        r = permutation_importance(m, X.iloc[validation], y_np[validation], scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=VARIABLES)
    print(f"{nom} : fait ({round(time.perf_counter() - debut)} s)", flush=True)

voix = sum(annonces[nom].astype(int) for nom in meilleurs)
DECISIONS = {**{nom: annonces[nom] for nom in meilleurs},
             "Vote à 2 voix sur 3 (règle retenue)": voix >= 2,
             "Unanimité (comparaison)": voix == 3,
             "Au moins un modèle (comparaison)": voix >= 1}

def moy_et(valeurs, facteur=100, decimales=1):
    v = np.asarray(valeurs, dtype=float)
    return f"{np.nanmean(v) * facteur:.{decimales}f} ± {np.nanstd(v) * facteur:.{decimales}f}"

lignes = []
for libelle_decision, a in DECISIONS.items():
    annoncees, retrouvees, precisions = [], [], []
    for _, validation in cv:
        av, yv = a[validation], y_np[validation]
        annoncees.append(int(av.sum()))
        retrouvees.append(yv[av].sum() / yv.sum())
        precisions.append(yv[av].mean() if av.any() else np.nan)
    ligne = {"Décision": libelle_decision, "Entrées annoncées (total)": int(a.sum()), "Vraies entrées parmi elles": int(y_np[a].sum()),
             "Entrées annoncées par pli": moy_et(annoncees, 1, 1), "Entrées retrouvées (%)": moy_et(retrouvees),
             "Précision des entrées annoncées (%)": moy_et(precisions)}
    if libelle_decision in meilleurs:
        ecarts, vrais = [], []
        for k, (_, validation) in enumerate(cv):
            p, s = probas[libelle_decision][validation], seuils[libelle_decision][k]
            ecarts += list(p[p >= s] - s)
            vrais += list(y_np[validation][p >= s])
        ordre = np.argsort(ecarts)[:FENETRE_MARGE]
        ligne[f"Précision marginale en validation, {FENETRE_MARGE} annoncés les plus proches du seuil (%)"] = round(np.mean(np.asarray(vrais)[ordre]) * 100, 1) if len(ordre) else np.nan
        ligne["Seuil (min - max)"] = f"{min(seuils[libelle_decision]):.3f} - {max(seuils[libelle_decision]):.3f}"
    lignes.append(ligne)
display(Markdown(f"**Entrées en retard annoncées en validation** ({len(y)} clients sains en août, dont {int(y.sum())} entrées en retard ; moyenne ± écart-type sur les 5 plis)"))
display(pd.DataFrame(lignes).set_index("Décision"))
display(pd.DataFrame({nom: {"Précision moyenne sur les entrées (%)": moy_et(mesures_auc[nom]["Précision moyenne"]),
                            "ROC AUC": moy_et(mesures_auc[nom]["ROC AUC"], 1, 3)} for nom in meilleurs}).T)

MLP : fait (208 s)
CatBoost : fait (72 s)
RandomForest : fait (16 s)


C:\Users\johan\AppData\Local\Temp\ipykernel_20640\3621449215.py:58: RuntimeWarning: Mean of empty slice
  return f"{np.nanmean(v) * facteur:.{decimales}f} ± {np.nanstd(v) * facteur:.{decimales}f}"
c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


**Entrées en retard annoncées en validation** (18978 clients sains en août, dont 799 entrées en retard ; moyenne ± écart-type sur les 5 plis)

,Entrées annoncées (total),Vraies entrées parmi elles,Entrées annoncées par pli,Entrées retrouvées (%),Précision des entrées annoncées (%),"Précision marginale en validation, 50 annoncés les plus proches du seuil (%)",Seuil (min - max)
Décision,,,,,,,
MLP,0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,inf - inf
CatBoost,0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,inf - inf
RandomForest,0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,inf - inf
Vote à 2 voix sur 3 (règle retenue),0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,NaN
Unanimité (comparaison),0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,NaN
Au moins un modèle (comparaison),0,0,0.0 ± 0.0,0.0 ± 0.0,nan ± nan,NaN,NaN


,Précision moyenne sur les entrées (%),ROC AUC
MLP,12.7 ± 1.1,0.724 ± 0.020
CatBoost,13.7 ± 0.7,0.771 ± 0.011
RandomForest,14.0 ± 1.5,0.769 ± 0.012


**Comment lire** :
- **Entrées retrouvées** : sur 100 vraies entrées en retard, combien sont annoncées. **Précision des entrées annoncées** : sur 100 annoncées, combien entrent réellement en retard. Avec une règle à la marge de 50 %, la précision de l'ensemble est plus haute que 50 % si les premiers annoncés sont des cas nets.
- **Précision marginale en validation** : elle doit tourner autour de 50 % ; mesurée sur 50 clients, elle est bruitée (de l'ordre de ± 7 points).
- **Seuil infini** (« inf ») sur un pli : même les 50 clients les plus probables n'atteignaient pas 50 % d'entrées dans les plis internes ; le modèle n'annonce alors personne sur ce pli.
- Les lignes « unanimité » et « au moins un modèle » situent la règle à 2 voix sur 3 ; elles ne servent pas à la choisir.

## 5. Accord des trois modèles

Pour chaque combinaison de voix, le nombre de clients et la part de vraies entrées en retard (septembre du train, prédictions hors pli).

In [6]:
noms = list(meilleurs)
combinaison = pd.Series([" + ".join(n for n in noms if annonces[n][i]) or "aucun" for i in range(len(y))])
table = pd.DataFrame({"Clients": combinaison.value_counts(), "Vraies entrées": pd.Series(y_np).groupby(combinaison).sum()})
table["Part de vraies entrées (%)"] = (table["Vraies entrées"] / table["Clients"] * 100).round(1)
display(table.sort_values("Clients", ascending=False))

,Clients,Vraies entrées,Part de vraies entrées (%)
aucun,18978,799,4.2


**Comment lire** : une ligne par groupe de modèles qui annoncent le client (« aucun » : aucun modèle ne l'annonce). Les groupes à deux ou trois modèles forment la décision commune.

## 6. L'importance des variables

Importance par permutation, sur chaque pli de validation, avec le modèle du pli : baisse de la précision moyenne sur les entrées en retard, en points (5 mélanges, moyenne sur les plis). Le second tableau regroupe les variables par mois.

In [7]:
table = pd.DataFrame(importances)
table["Moyenne des modèles"] = table.mean(axis=1)
display(Markdown("**Par variable**"))
display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
NOMS_MOIS = {1: "septembre (M)", 2: "août (M-1)", 3: "juillet (M-2)", 4: "juin (M-3)", 5: "mai (M-4)", 6: "avril (M-5)"}
def mois_de(variable):
    return "plafond" if variable == "LIMIT_BAL" else NOMS_MOIS[int(variable[-1])]
display(Markdown("**Par mois**"))
display(table.groupby(table.index.map(mois_de)).sum().reindex([NOMS_MOIS[n] for n in range(1, 7)] + ["plafond"]).round(2))

**Par variable**

,MLP,CatBoost,RandomForest,Moyenne des modèles
PAY_3,1.78,3.37,2.81,2.65
LIMIT_BAL,1.04,2.32,1.90,1.75
PAY_5,1.34,0.96,1.55,1.28
PAY_6,1.12,0.79,0.88,0.93
BILL_AMT2,0.62,1.99,0.16,0.92
PAY_4,0.55,0.98,0.63,0.72
BILL_AMT1,0.34,1.52,0.13,0.66
PAY_AMT1,0.23,0.95,0.48,0.55
PAY_AMT2,0.71,0.58,0.31,0.53
PAY_2,0.31,0.61,0.33,0.42


**Par mois**

,MLP,CatBoost,RandomForest,Moyenne des modèles
septembre (M),0.57,2.47,0.61,1.21
août (M-1),1.63,3.18,0.80,1.87
juillet (M-2),2.35,3.84,3.04,3.08
juin (M-3),1.15,1.36,0.98,1.16
mai (M-4),1.86,0.80,1.56,1.40
avril (M-5),1.69,1.31,1.20,1.40
plafond,1.04,2.32,1.90,1.75


**Comment lire** : proche de 0, le modèle ne s'en sert pas.

## 7. Application aux `PAY_1` = 1 sains en août

Chaque modèle est appris sur **tous les clients sains en août du train** avec son meilleur réglage ; son seuil est fixé sur ses probabilités hors pli (section 4), à 50 % de précision marginale. Les trois modèles sont appliqués aux clients codifiés 1 en septembre et sains en août ; la décision commune est le vote à 2 voix sur 3. Aucun fichier de données n'est modifié ; les modèles et leurs seuils sont enregistrés dans `modeles/11d/`.

In [8]:
import json, platform
from datetime import datetime
import catboost, sklearn
from joblib.externals import cloudpickle

X_1 = a_trancher.loc[a_trancher["PAY_2"] <= 0, VARIABLES].copy()
for col in CATEGORIELLES:
    X_1[col] = regrouper(X_1[col])
dossier_modeles = Path.cwd() / "modeles" / "11d"
dossier_modeles.mkdir(parents=True, exist_ok=True)
FICHIERS = {"MLP": "mlp", "CatBoost": "catboost", "RandomForest": "randomforest"}
synthese = {"scenario": "11d", "date": datetime.now().isoformat(timespec="seconds"),
            "source": "data/creditcard_pret_ingestion.csv (niveau 0), sans dpnm, sans ID, sans données personnelles, sans correctifs",
            "exemples": "clients du train sains en août (-2, -1, 0), septembre seul, historique complet",
            "cible": "1 = entrée en retard en septembre (PAY_1 >= 2) ; 0 = reste sain",
            "variables": VARIABLES, "critere": "précision moyenne sur les entrées en retard",
            "regle_de_decision": f"précision marginale de {PRECISION_MARGINALE_VISEE:.0%} sur les {FENETRE_MARGE} derniers annoncés, vote à 2 voix sur 3",
            "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
            "modeles": {}}
annonce_1, lignes = {}, []
for nom, params in meilleurs.items():
    seuil = seuil_marginal(y_np, probas[nom])
    modele = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X, y_np)
    with open(dossier_modeles / f"11d_{FICHIERS[nom]}.pkl", "wb") as f:
        cloudpickle.dump(modele, f)
    p1 = modele.predict_proba(X_1)[:, 1]
    annonce_1[nom] = p1 >= seuil
    synthese["modeles"][nom] = {"fichier": f"11d_{FICHIERS[nom]}.pkl", "seuil": float(seuil),
                                "reglage": {k.split("__")[-1]: (v if isinstance(v, (int, float, str)) or v is None else str(v)) for k, v in params.items()}}
    lignes.append({"Modèle": nom, "Seuil": round(float(seuil), 3),
                   "Train : part annoncée en retard (%)": round((probas[nom] >= seuil).mean() * 100, 2),
                   f"PAY_1 = 1 sains en août ({len(X_1)}) : annoncés en retard": int(annonce_1[nom].sum()),
                   "Part annoncée en retard (%)": round(annonce_1[nom].mean() * 100, 2),
                   "Probabilité médiane, train": round(float(np.median(probas[nom])), 4),
                   "Probabilité médiane, PAY_1 = 1": round(float(np.median(p1)), 4)})
(dossier_modeles / "11d_synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")
display(pd.DataFrame(lignes).set_index("Modèle"))
voix_1 = sum(annonce_1[nom].astype(int) for nom in meilleurs)
print(f"PAY_1 = 1 sains en août annoncés en retard, vote à 2 voix sur 3 : {int((voix_1 >= 2).sum())} sur {len(X_1)} "
      f"(unanimité : {int((voix_1 == 3).sum())} ; au moins un modèle : {int((voix_1 >= 1).sum())})")
print(f"Modèles et seuils enregistrés dans {dossier_modeles}")

,Seuil,Train : part annoncée en retard (%),PAY_1 = 1 sains en août (1836) : annoncés en retard,Part annoncée en retard (%),"Probabilité médiane, train","Probabilité médiane, PAY_1 = 1"
Modèle,,,,,,
MLP,inf,0.0,0,0.0,0.0363,0.0022
CatBoost,inf,0.0,0,0.0,0.3752,0.0132
RandomForest,inf,0.0,0,0.0,0.0351,0.0071


PAY_1 = 1 sains en août annoncés en retard, vote à 2 voix sur 3 : 0 sur 1836 (unanimité : 0 ; au moins un modèle : 0)
Modèles et seuils enregistrés dans c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\modeles\11d


**Comment lire** : la part annoncée en retard se compare entre le train et les `PAY_1` = 1. Les 1 sains en août sont presque tous à -2 ou -1 en août (section 1), des codifications d'où l'on entre rarement en retard : peu d'annonces sont attendues chez eux.

## 8. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*